In [1]:
import sys
import pandas as pd
import numpy as np
from pathlib import Path

REPO_ROOT    = Path("..").resolve()
SCRIPTS_DIR  = REPO_ROOT / "scripts"
RESULTS_DIR  = REPO_ROOT / "results"
DATA_PATH    = REPO_ROOT / "data" / "final_data_subsetFLGAAL8.npz"
ADBENCH_DIR  = REPO_ROOT / "data" / "adbench"

# Appendix A results go in their own subfolder
APPENDIX_A_DIR = RESULTS_DIR / "appendix_a"

sys.path.insert(0, str(SCRIPTS_DIR))

print("Repo root      :", REPO_ROOT)
print("BLS data       :", DATA_PATH, "| exists:", DATA_PATH.exists())
print("ADBench dir    :", ADBENCH_DIR, "| exists:", ADBENCH_DIR.exists())
print("Results dir    :", RESULTS_DIR)
print("Appendix A dir :", APPENDIX_A_DIR)

if ADBENCH_DIR.exists():
    print(f"ADBench datasets found: {len(list(ADBENCH_DIR.glob('*.npz')))}")


Repo root      : C:\Users\jrfal\OneDrive\Documents\Master and Doctor\GeorgeWashintonUni\PRAXIS\Paper_Journal_Post_Praxis\GITHUB
BLS data       : C:\Users\jrfal\OneDrive\Documents\Master and Doctor\GeorgeWashintonUni\PRAXIS\Paper_Journal_Post_Praxis\GITHUB\data\final_data_subsetFLGAAL8.npz | exists: True
ADBench dir    : C:\Users\jrfal\OneDrive\Documents\Master and Doctor\GeorgeWashintonUni\PRAXIS\Paper_Journal_Post_Praxis\GITHUB\data\adbench | exists: True
Results dir    : C:\Users\jrfal\OneDrive\Documents\Master and Doctor\GeorgeWashintonUni\PRAXIS\Paper_Journal_Post_Praxis\GITHUB\results
Appendix A dir : C:\Users\jrfal\OneDrive\Documents\Master and Doctor\GeorgeWashintonUni\PRAXIS\Paper_Journal_Post_Praxis\GITHUB\results\appendix_a
ADBench datasets found: 29


In [2]:
# RUN_BLS_TRAINING     = True  re-runs Table 5  (~2-3 hours on GPU)
# RUN_ADBENCH_TRAINING = True  re-runs Appendix A (~8 hours on GPU)
# Both False by default — loads pre-computed results instantly

RUN_BLS_TRAINING     = True
RUN_ADBENCH_TRAINING = True

# Seeds and methods — must match paper exactly
SEEDS   = [69, 72, 888]
METHODS = [
    "KNN", "LOF", "IForest", "PCA", "CBLOF", "HBOS",
    "ECOD", "COPOD", "LODA", "DeepSVDD",
    "CALDM-D", "CALDM-J",
]
BLS_SUBSET   = 0.20   # 20% stratified sub-sample used in paper
VAE_EPOCHS   = 15
DIFF_EPOCHS  = 200
VAE_MAX_BETA = 0.1


In [3]:
if RUN_BLS_TRAINING:
    if not DATA_PATH.exists():
        raise FileNotFoundError(
            f"\nBLS dataset not found at:\n  {DATA_PATH}\n\n"
            "Please download from: https://data.hrmatica.com/wagescale"
        )

    from caldm_benchmark import run_bls_benchmark

    RESULTS_DIR.mkdir(parents=True, exist_ok=True)

    print(f"Running BLS benchmark — {len(METHODS)} methods x {len(SEEDS)} seeds")
    print(f"Sub-sample: {BLS_SUBSET*100:.0f}%  |  VAE epochs: {VAE_EPOCHS}  "
          f"|  Diffusion epochs: {DIFF_EPOCHS}  |  β: {VAE_MAX_BETA}")
    print("Estimated time: 2–3 hours on GPU.\n")

    run_bls_benchmark(
        data_path    = str(DATA_PATH),
        subset_pct   = BLS_SUBSET,
        seeds        = SEEDS,
        methods      = METHODS,
        vae_epochs   = VAE_EPOCHS,
        diff_epochs  = DIFF_EPOCHS,
        vae_max_beta = VAE_MAX_BETA,
        output_dir   = str(RESULTS_DIR),
        resume       = False,
    )
    print("BLS benchmark complete. Results saved to:", RESULTS_DIR)

else:
    print("RUN_BLS_TRAINING = False — loading pre-computed BLS results.")


Running BLS benchmark — 12 methods x 3 seeds
Sub-sample: 20%  |  VAE epochs: 15  |  Diffusion epochs: 200  |  β: 0.1
Estimated time: 2–3 hours on GPU.

[setup] device=cuda  seeds=[69, 72, 888]  subset_pct=0.2
[setup] methods (12): KNN, LOF, IForest, PCA, CBLOF, HBOS, ECOD, COPOD, LODA, DeepSVDD, CALDM-D, CALDM-J
[setup] writing rows incrementally to C:\Users\jrfal\OneDrive\Documents\Master and Doctor\GeorgeWashintonUni\PRAXIS\Paper_Journal_Post_Praxis\GITHUB\results\bls_per_run.csv
  [load_bls] X.shape=(596856, 202)  y.shape=(596856, 2)  y.dtype=object  y[0]=array([0, 'NONE'], dtype=object)
  [load_bls] format detected: 2D strings (n,2) — col 0=binary, col 1=type
  [load_bls] outlier rate: 0.0739  type codes: [0, 1, 2, 3, 4, 5]

=== seed=69 ===
  train=95495  test=23874  features=202  outlier_rate_test=0.0739
  label format detected: 2D strings (n,2) — col 0=binary, col 1=type
  CALDM hp: {'hidden_dim': 256, 'latent_dim': 32, 'context_dim': 40, 'attention_heads': 4, 'batch_size': 256}


In [4]:
if RUN_ADBENCH_TRAINING:
    if not ADBENCH_DIR.exists() or not list(ADBENCH_DIR.glob("*.npz")):
        raise FileNotFoundError(
            f"\nADBench datasets not found at:\n  {ADBENCH_DIR}\n\n"
            "Download from: https://github.com/Minqi824/ADBench\n"
            "Place all .npz files in data/adbench/"
        )

    from caldm_benchmark import run_adbench_benchmark

    APPENDIX_A_DIR.mkdir(parents=True, exist_ok=True)

    print(f"Running ADBench auto-scaled benchmark — {len(METHODS)} methods x {len(SEEDS)} seeds")
    print(f"VAE epochs: {VAE_EPOCHS}  |  Diffusion epochs: {DIFF_EPOCHS}  |  β: {VAE_MAX_BETA}")
    print("Estimated time: 6–8 hours on GPU.\n")

    run_adbench_benchmark(
        input_dir    = str(ADBENCH_DIR),
        seeds        = SEEDS,
        methods      = METHODS,
        vae_epochs   = VAE_EPOCHS,
        diff_epochs  = DIFF_EPOCHS,
        vae_max_beta = VAE_MAX_BETA,
        output_dir   = str(APPENDIX_A_DIR),
        resume       = True,   # safe to resume if interrupted
    )
    print("ADBench benchmark complete. Results saved to:", APPENDIX_A_DIR)

else:
    print("RUN_ADBENCH_TRAINING = False — loading pre-computed ADBench results.")


Running ADBench auto-scaled benchmark — 12 methods x 3 seeds
VAE epochs: 15  |  Diffusion epochs: 200  |  β: 0.1
Estimated time: 6–8 hours on GPU.

[setup] device=cuda  seeds=[69, 72, 888]
[setup] datasets: 29  methods: 12
[setup] writing rows incrementally to C:\Users\jrfal\OneDrive\Documents\Master and Doctor\GeorgeWashintonUni\PRAXIS\Paper_Journal_Post_Praxis\GITHUB\results\appendix_a\adbench_per_run.csv

[1/29] 10_cover  (286048 rows, 10 features, 0.96% outliers)
  [run] KNN seed=69   AUC=0.804  AP=0.041  (80.2s)
  [run] KNN seed=72   AUC=0.804  AP=0.041  (79.6s)
  [run] KNN seed=888   AUC=0.804  AP=0.041  (79.2s)
  [run] LOF seed=69   AUC=0.520  AP=0.012  (51.6s)
  [run] LOF seed=72   AUC=0.520  AP=0.012  (49.2s)
  [run] LOF seed=888   AUC=0.520  AP=0.012  (49.1s)
  [run] IForest seed=69   AUC=0.887  AP=0.057  (3.4s)
  [run] IForest seed=72   AUC=0.888  AP=0.052  (3.5s)
  [run] IForest seed=888   AUC=0.838  AP=0.034  (3.4s)
  [run] PCA seed=69   AUC=0.935  AP=0.073  (0.5s)
  [run]

In [5]:
bls_path = RESULTS_DIR / "bls_summary.csv"
adb_path = APPENDIX_A_DIR / "adbench_summary.csv"

if not bls_path.exists():
    raise FileNotFoundError(
        f"BLS results not found at {bls_path}.\n"
        "Set RUN_BLS_TRAINING = True in Cell 2 to generate them."
    )
if not adb_path.exists():
    raise FileNotFoundError(
        f"ADBench results not found at {adb_path}.\n"
        "Set RUN_ADBENCH_TRAINING = True in Cell 2 to generate them."
    )

bls_summary = pd.read_csv(bls_path)
adb_summary = pd.read_csv(adb_path)

print(f"BLS summary     : {len(bls_summary)} methods")
print(f"ADBench summary : {len(adb_summary)} methods  "
      f"({adb_summary['n_datasets'].iloc[0]:.0f} datasets)")


# ============================================================
# CELL 6 — Reproduce Table 5 (BLS benchmark)
# ============================================================

def fmt(mean, std, d=3):
    return f"{mean:.{d}f} \u00b1 {std:.{d}f}"

rows = []
for _, r in bls_summary.sort_values("auc_roc_mean", ascending=False).iterrows():
    rows.append({
        "Method": r["method"],
        "AUROC":  fmt(r["auc_roc_mean"], r["auc_roc_std"]),
        "AP":     fmt(r["ap_mean"],       r["ap_std"]),
        "F1":     fmt(r["f1_mean"],       r["f1_std"]),
    })

table5 = pd.DataFrame(rows)
print("Table 5. BLS Compensation Dataset \u2014 Performance Results")
print("(all 12 methods, same 20% sub-sample, seeds [69, 72, 888])\n")
print(table5.to_string(index=False))


# ============================================================
# CELL 7 — Verify key BLS claims from the paper
# ============================================================

caldm_d = bls_summary[bls_summary["method"] == "CALDM-D"].iloc[0]
knn     = bls_summary[bls_summary["method"] == "KNN"].iloc[0]
cblof   = bls_summary[bls_summary["method"] == "CBLOF"].iloc[0]
caldm_j = bls_summary[bls_summary["method"] == "CALDM-J"].iloc[0]

print("=" * 62)
print("Table 5 paper claims verification")
print("=" * 62)

print(f"\nCALDM-D AUROC : {caldm_d['auc_roc_mean']:.3f} \u00b1 "
      f"{caldm_d['auc_roc_std']:.3f}   paper: 0.740 \u00b1 0.016")
print(f"KNN     AUROC : {knn['auc_roc_mean']:.3f} \u00b1 "
      f"{knn['auc_roc_std']:.3f}   paper: 0.757 \u00b1 0.003")

auroc_gap = (knn["auc_roc_mean"] - caldm_d["auc_roc_mean"]) * 100
print(f"AUROC gap (KNN - CALDM-D): {auroc_gap:.1f} pp   paper: 1.7 pp")

print(f"\nCALDM-D AP : {caldm_d['ap_mean']:.3f}   paper: 0.313")
print(f"CBLOF   AP : {cblof['ap_mean']:.3f}   paper: 0.280")
ap_adv = (caldm_d["ap_mean"] - cblof["ap_mean"]) / cblof["ap_mean"] * 100
print(f"CALDM-D AP advantage over CBLOF: {ap_adv:.1f}%   paper: +11.7%")

f1_adv = (caldm_d["f1_mean"] - knn["f1_mean"]) / knn["f1_mean"] * 100
print(f"CALDM-D F1 advantage over KNN  : {f1_adv:.1f}%   paper: +16.6%")

print(f"\nCALDM-J AUROC std : {caldm_j['auc_roc_std']:.3f}   paper: 0.027")
print(f"CALDM-D AUROC std : {caldm_d['auc_roc_std']:.3f}   paper: 0.016")
print(f"Variance ratio J/D: "
      f"{caldm_j['auc_roc_std']/caldm_d['auc_roc_std']:.1f}x   paper: 1.7x")


# ============================================================
# CELL 8 — Reproduce Appendix A Table A.1 (auto-scaled ADBench)
# ============================================================

rows = []
for _, r in adb_summary.sort_values("auc_roc_mean", ascending=False).iterrows():
    rows.append({
        "Method":       r["method"],
        "Mean AUROC":   fmt(r["auc_roc_mean"], r["auc_roc_std"]),
        "Mean AP":      fmt(r["ap_mean"],       r["ap_std"]),
        "AUROC wins":   f"{int(r['wins_auc_roc'])} / {int(r['n_datasets'])}",
        "AP wins":      f"{int(r['wins_ap'])} / {int(r['n_datasets'])}",
    })

tableA1 = pd.DataFrame(rows)
n_datasets = int(adb_summary["n_datasets"].iloc[0])
print(f"Table A.1. ADBench Auto-Scaled Benchmark ({n_datasets} datasets, "
      f"12 methods, seeds {SEEDS})\n")
print(tableA1.to_string(index=False))


# ============================================================
# CELL 9 — Verify key Appendix A claims
# ============================================================

caldm_d_adb = adb_summary[adb_summary["method"] == "CALDM-D"].iloc[0]
iforest_adb = adb_summary[adb_summary["method"] == "IForest"].iloc[0]
caldm_j_adb = adb_summary[adb_summary["method"] == "CALDM-J"].iloc[0]

print("=" * 62)
print("Appendix A paper claims verification")
print("=" * 62)

print(f"\nCALDM-D AUROC : {caldm_d_adb['auc_roc_mean']:.4f}   paper: 0.7834")
print(f"IForest AUROC : {iforest_adb['auc_roc_mean']:.4f}   paper: 0.7925")
gap = iforest_adb["auc_roc_mean"] - caldm_d_adb["auc_roc_mean"]
print(f"Gap IForest - CALDM-D: {gap:.4f}   paper: 0.009")

# Rank of CALDM-D
ranked = adb_summary.sort_values("auc_roc_mean", ascending=False).reset_index(drop=True)
caldm_d_rank = ranked[ranked["method"] == "CALDM-D"].index[0] + 1
print(f"\nCALDM-D rank (AUROC) : {caldm_d_rank} / {len(ranked)}   paper: 2 / 12")

# CALDM-D vs CALDM-J gap
adb_gap = caldm_d_adb["auc_roc_mean"] - caldm_j_adb["auc_roc_mean"]
print(f"\nCALDM-D AUROC: {caldm_d_adb['auc_roc_mean']:.4f}")
print(f"CALDM-J AUROC: {caldm_j_adb['auc_roc_mean']:.4f}")
print(f"Gap D - J    : {adb_gap:.3f}   paper: 0.031")

# Variance ratio on BLS (cross-referenced from Cell 7)
print(f"\nCALDM-J AP std: {caldm_j_adb['ap_std']:.3f}   paper: 0.056")
print(f"CALDM-D AP std: {caldm_d_adb['ap_std']:.3f}   paper: 0.013")
ap_var_ratio = caldm_j_adb["ap_std"] / caldm_d_adb["ap_std"]
print(f"AP variance ratio J/D: {ap_var_ratio:.1f}x   paper: 4.4x")

BLS summary     : 12 methods
ADBench summary : 12 methods  (27 datasets)
Table 5. BLS Compensation Dataset — Performance Results
(all 12 methods, same 20% sub-sample, seeds [69, 72, 888])

  Method         AUROC            AP            F1
     KNN 0.757 ± 0.003 0.243 ± 0.008 0.246 ± 0.012
 CALDM-D 0.740 ± 0.016 0.313 ± 0.013 0.288 ± 0.010
   CBLOF 0.717 ± 0.007 0.280 ± 0.027 0.275 ± 0.021
    ECOD 0.698 ± 0.008 0.148 ± 0.005 0.205 ± 0.011
   COPOD 0.679 ± 0.008 0.139 ± 0.005 0.193 ± 0.015
 CALDM-J 0.679 ± 0.027 0.212 ± 0.056 0.226 ± 0.073
     LOF 0.645 ± 0.005 0.096 ± 0.001 0.047 ± 0.004
    HBOS 0.640 ± 0.008 0.117 ± 0.006 0.148 ± 0.013
 IForest 0.630 ± 0.012 0.117 ± 0.010 0.143 ± 0.020
     PCA 0.572 ± 0.006 0.087 ± 0.002 0.083 ± 0.003
DeepSVDD 0.534 ± 0.020 0.081 ± 0.004 0.084 ± 0.006
    LODA 0.517 ± 0.028 0.079 ± 0.006 0.084 ± 0.015
Table 5 paper claims verification

CALDM-D AUROC : 0.740 ± 0.016   paper: 0.740 ± 0.016
KNN     AUROC : 0.757 ± 0.003   paper: 0.757 ± 0.003
AUROC g